# RSNA Knee Abnormality Detection — Phase 4B-2A
## DINOv3-inspired MRI representation + public-checkpoint provenance audit

### Why this phase exists

The completed clean MRI baseline established that the MRI pipeline works, but the ResNet18 SliceMIL representation is intentionally simple:

```text
slice → ResNet18 → mean/max MIL
      → mean within plane
      → equal mean across planes
```

The uploaded high-scoring public ensemble uses a much richer representation:

```text
plane × fat-suppression slots
        ↓
ordered multi-slice input
        ↓
strong self-supervised encoder
        ↓
series-type embeddings
        ↓
target-specific cross-series attention
        ↓
12 findings
```

The public notebook also explicitly warns that later ensemble/blend revisions were tuned to tiny public-leaderboard changes. Therefore we copy **architectural principles**, not leaderboard-tuned blend constants.

### LLM Council synthesis

**Contrarian**
Do not use the five public competition-fine-tuned DINOv3 checkpoints as a supposedly clean gold benchmark unless their train/holdout UID provenance is verifiable.

**First Principles**
The next bottleneck is not the pseudo-label pipeline. It is how a whole multi-series MRI study is represented for twelve different findings.

**Expansionist**
The strongest reusable ideas are:
- 6 anatomical/contrast slots,
- physical field-of-view normalization,
- neighboring-slice context,
- series-type embeddings,
- target-specific attention.

**Outsider**
A ~0.92 public leaderboard ensemble is not one model. It combines many folds, families, rank transforms, and calibration steps. We must decompose it.

**Executor**
Before spending GPU-hours, audit:
1. whether the public DINO checkpoints are provenance-safe,
2. whether our dataset supports the 6-slot design,
3. whether 130 mm physical crops are feasible,
4. whether 16-slice windows are available.

### This notebook does NOT train a new model

It produces the frozen input specification for **Phase 4B-2B**, our clean DINOv3-style training experiment.

In [ ]:
from pathlib import Path
from hashlib import sha256
import json, warnings, re

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_colwidth", 300)

EXPERIMENT_VERSION = "PHASE4B2A_DINOV3_REPRESENTATION_PROVENANCE_AUDIT_v1"

CROP_MM = 130.0
MODEL_SIZE = 336
SLICE_BAND = (0.12, 0.88)
N_SLICES = 16

SLOTS = [
    ("Sagittal", 1, "SAG_FS"),
    ("Sagittal", 0, "SAG_NONFS"),
    ("Coronal", 1, "COR_FS"),
    ("Coronal", 0, "COR_NONFS"),
    ("Axial", 1, "AX_FS"),
    ("Axial", 0, "AX_NONFS"),
]

print("Experiment:", EXPERIMENT_VERSION)
print("Crop:", CROP_MM, "mm")
print("Image size:", MODEL_SIZE)
print("Slice band:", SLICE_BAND)
print("Slices per slot:", N_SLICES)

In [ ]:
# ============================================================
# EXPLICIT KAGGLE PATHS — USER DATASETS
# ============================================================

PHASE4B1A_DIR = Path(
    "/kaggle/input/datasets/aymenslimani/4b-1a"
)

PHASE4B1B_V1_DIR = Path(
    "/kaggle/input/datasets/aymenslimani/result-phase-4-b"
)

PIPELINE_JSON_PATH = PHASE4B1A_DIR / "PHASE4B1A_MRI_PIPELINE_v1.json"
SERIES_INDEX_PATH = PHASE4B1A_DIR / "rsna_phase4b1a_train_series_index.csv"
SLICE_SELECTION_PATH = PHASE4B1A_DIR / "rsna_phase4b1a_baseline_slice_selection.parquet"
STUDY_INDEX_PATH = PHASE4B1A_DIR / "rsna_phase4b1a_study_index.csv"

BASELINE_V1_JSON_PATH = (
    PHASE4B1B_V1_DIR
    / "PHASE4B1B_FIRST_MRI_BASELINE_v1_1.json"
)

required_paths = [
    PIPELINE_JSON_PATH,
    SERIES_INDEX_PATH,
    SLICE_SELECTION_PATH,
    STUDY_INDEX_PATH,
    BASELINE_V1_JSON_PATH,
]

print("=== Explicit Kaggle paths ===")

for p in required_paths:
    print(("✅" if p.exists() else "❌"), p)

missing_paths = [
    p for p in required_paths
    if not p.exists()
]

if missing_paths:
    raise FileNotFoundError(
        "Missing Kaggle input(s):\n"
        + "\n".join(str(p) for p in missing_paths)
        + "\n\nAttach both the 4b-1a and result-phase-4-b datasets."
    )

print("\n✅ All explicit Kaggle paths exist")

## 1. Locate Kaggle inputs automatically

In [ ]:
# ============================================================
# ROBUST CHECKPOINT DISCOVERY
# ============================================================
# The earlier notebook searched only for m_f0.pt ... m_f4.pt.
# Kaggle may mount/rename files differently, so this version:
#   1) inventories every .pt/.pth/.ckpt file under /kaggle/input
#   2) scores likely knee-MRI/DINO/fold checkpoints
#   3) audits likely candidates
#   4) DOES NOT fail if no public checkpoint is found
#
# Public competition-trained checkpoints are never treated as
# clean validation models unless train/validation UID provenance
# is independently verifiable.

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/kaggle/working"),
]

CHECKPOINT_SUFFIXES = {".pt", ".pth", ".ckpt"}

def infer_fold_from_name(path):
    s = str(path).lower()

    patterns = [
        r"(?:^|[_\-/])m[_-]?f(?:old)?[_-]?(\d+)",
        r"(?:^|[_\-/])fold[_-]?(\d+)",
        r"(?:^|[_\-/])f[_-]?(\d+)(?:[_.\-/]|$)",
    ]

    for pat in patterns:
        m = re.search(pat, s)

        if m:
            return int(m.group(1))

    return None


def checkpoint_keyword_score(path):
    s = str(path).lower()

    score = 0

    weights = {
        "dino": 6,
        "dinov3": 8,
        "vit": 3,
        "knee": 4,
        "mri": 4,
        "fold": 3,
        "m_f": 3,
        "weight": 1,
    }

    for key, w in weights.items():
        if key in s:
            score += w

    if infer_fold_from_name(path) is not None:
        score += 4

    return score


checkpoint_rows = []

for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    try:
        files = [
            p
            for p in root.rglob("*")
            if p.is_file()
            and p.suffix.lower() in CHECKPOINT_SUFFIXES
        ]
    except Exception:
        files = []

    for p in files:

        try:
            size_mb = (
                p.stat().st_size
                / (1024 ** 2)
            )
        except Exception:
            size_mb = np.nan

        checkpoint_rows.append(
            {
                "path": str(p),
                "filename": p.name,
                "suffix": p.suffix.lower(),
                "size_mb": size_mb,
                "inferred_fold": infer_fold_from_name(p),
                "keyword_score": checkpoint_keyword_score(p),
            }
        )


checkpoint_inventory = pd.DataFrame(
    checkpoint_rows,
    columns=[
        "path",
        "filename",
        "suffix",
        "size_mb",
        "inferred_fold",
        "keyword_score",
    ],
)

if len(checkpoint_inventory):
    checkpoint_inventory = (
        checkpoint_inventory
        .sort_values(
            [
                "keyword_score",
                "size_mb",
                "path",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


print(
    "Checkpoint files found under Kaggle inputs:",
    len(checkpoint_inventory),
)

if len(checkpoint_inventory):
    display(
        checkpoint_inventory.head(50)
    )


# ------------------------------------------------------------
# Candidate selection
# ------------------------------------------------------------
# Strong preference for files with DINO / knee / MRI / fold clues.
# We do NOT blindly use these weights for evaluation.
candidate_inventory = checkpoint_inventory.loc[
    checkpoint_inventory[
        "keyword_score"
    ] >= 4
].copy()


# If a knee-mri-fold-weights directory exists, prioritize it.
if len(candidate_inventory):
    preferred_mask = (
        candidate_inventory["path"]
        .str.lower()
        .str.contains(
            "knee.*mri.*fold|knee-mri-fold|knee_mri_fold",
            regex=True,
            na=False,
        )
    )

    if preferred_mask.any():
        candidate_inventory = (
            candidate_inventory
            .loc[preferred_mask]
            .copy()
        )


# Limit checkpoint loading to a reasonable number.
candidate_inventory = (
    candidate_inventory
    .head(20)
    .copy()
)


PUBLIC_CHECKPOINT_PATHS = [
    (
        row.inferred_fold
        if pd.notna(row.inferred_fold)
        else None,
        Path(row.path),
    )
    for row in candidate_inventory.itertuples(
        index=False
    )
]


print()
print(
    "Likely public DINO/knee-MRI checkpoint candidates:",
    len(PUBLIC_CHECKPOINT_PATHS),
)

for fold, p in PUBLIC_CHECKPOINT_PATHS:
    print(
        f"fold={fold} | {p}"
    )


if not PUBLIC_CHECKPOINT_PATHS:
    print()
    print(
        "⚠️ No likely public DINO checkpoint was identified."
    )
    print(
        "This is NOT a blocker for Phase 4B-2A."
    )
    print(
        "The representation audit will continue and provenance will be marked unavailable."
    )


OUT_DIR = (
    Path("/kaggle/working")
    if Path("/kaggle/working").exists()
    else Path("/mnt/data")
)

print()
print(
    "Output directory:",
    OUT_DIR,
)

## 2. Validate upstream Phase 4B-1A artifacts

In [ ]:
def file_sha256(path, block=1024*1024):
    h = sha256()

    with open(path, "rb") as f:
        while True:
            b = f.read(block)

            if not b:
                break

            h.update(b)

    return h.hexdigest()


with open(
    PIPELINE_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:
    pipeline_meta = json.load(f)


with open(
    BASELINE_V1_JSON_PATH,
    "r",
    encoding="utf-8",
) as f:
    baseline_v1_meta = json.load(f)


series = pd.read_csv(
    SERIES_INDEX_PATH
)

study_index = pd.read_csv(
    STUDY_INDEX_PATH
)

slices = pd.read_parquet(
    SLICE_SELECTION_PATH
)


series["StudyInstanceUID"] = (
    series["StudyInstanceUID"].astype(str)
)

series["SeriesInstanceUID"] = (
    series["SeriesInstanceUID"].astype(str)
)

study_index["StudyInstanceUID"] = (
    study_index["StudyInstanceUID"].astype(str)
)

slices["StudyInstanceUID"] = (
    slices["StudyInstanceUID"].astype(str)
)

slices["SeriesInstanceUID"] = (
    slices["SeriesInstanceUID"].astype(str)
)


required_series = {
    "StudyInstanceUID",
    "SeriesInstanceUID",
    "Anatomical_Plane",
    "Fat_Suppression",
    "n_readable",
    "qc_hard_fail",
}

missing = required_series - set(series.columns)

assert not missing, (
    f"Series index missing: {missing}"
)


print("=== Phase 4B-1A ===")
print(
    "Pipeline:",
    pipeline_meta.get(
        "experiment_version",
        pipeline_meta.get("pipeline_version"),
    ),
)

print(
    "Study index studies:",
    study_index["StudyInstanceUID"].nunique(),
)

print("Series rows:", len(series))
print("Slice selection rows:", len(slices))


print("\n=== Previous Phase 4B-1B v1.1 baseline ===")
print(
    "Experiment:",
    baseline_v1_meta.get("experiment_version"),
)

print(
    "Gold macro ROC-AUC:",
    baseline_v1_meta.get(
        "gold_results",
        {},
    ).get("macro_roc_auc"),
)

assert (
    baseline_v1_meta.get("upstream_pipeline")
    == "PHASE4B1A_MRI_INDEX_ORDER_SELECT_LOADER_QC_v1"
)

print("\n✅ Upstream files and baseline provenance loaded")

## 3. Build the 6-slot study representation audit

In [ ]:
def as_binary(v):
    if pd.isna(v):
        return 0
    if isinstance(v, (int, float, np.integer, np.floating)):
        return int(float(v) > 0)
    s = str(v).strip().lower()
    return int(s in {"1", "true", "yes", "y", "fs", "fatsat", "fat_suppression"})

series["fat_bin"] = series["Fat_Suppression"].map(as_binary)

slot_map = {(plane, fs): slot_name for plane, fs, slot_name in SLOTS}

series["slot"] = [
    slot_map.get((str(p), int(fs)), "OTHER")
    for p, fs in zip(series["Anatomical_Plane"], series["fat_bin"])
]

eligible_series = series.loc[
    ~series["qc_hard_fail"].fillna(True).astype(bool)
].copy()

if "Fluid_Sensitive" in eligible_series.columns:
    eligible_series["fluid_bin"] = eligible_series["Fluid_Sensitive"].map(as_binary)
else:
    eligible_series["fluid_bin"] = 0

eligible_series = eligible_series.sort_values(
    ["StudyInstanceUID", "slot", "n_readable", "fluid_bin", "SeriesInstanceUID"],
    ascending=[True, True, False, False, True],
)

slot_plan = (
    eligible_series.loc[eligible_series["slot"].ne("OTHER")]
    .groupby(["StudyInstanceUID", "slot"], as_index=False)
    .first()
)

slot_presence = (
    slot_plan.assign(present=1)
    .pivot(index="StudyInstanceUID", columns="slot", values="present")
    .fillna(0)
)

for _, _, slot_name in SLOTS:
    if slot_name not in slot_presence.columns:
        slot_presence[slot_name] = 0

slot_presence = slot_presence[[slot_name for _, _, slot_name in SLOTS]]
slot_presence["n_slots"] = slot_presence.sum(axis=1)

print(slot_presence["n_slots"].value_counts().sort_index())
print()
print("Mean slots/study:", slot_presence["n_slots"].mean())
print("Studies with all 6 slots:", int((slot_presence["n_slots"] == 6).sum()))
print("Studies with >=4 slots:", int((slot_presence["n_slots"] >= 4).sum()))

In [ ]:
slot_coverage_rows = []
n_studies = series["StudyInstanceUID"].nunique()

for _, _, slot_name in SLOTS:
    n = int(slot_presence[slot_name].sum())
    slot_coverage_rows.append({
        "slot": slot_name,
        "n_studies_present": n,
        "fraction_studies_present": n / n_studies,
    })

slot_coverage = pd.DataFrame(slot_coverage_rows)
display(slot_coverage)

## 4. Audit 16-slice central-band feasibility

In [ ]:
def band_count(n):
    n = int(max(n, 0))
    if n == 0:
        return 0
    lo = int(np.floor(SLICE_BAND[0] * (n - 1)))
    hi = int(np.ceil(SLICE_BAND[1] * (n - 1)))
    return max(hi - lo + 1, 0)

slot_plan["central_band_count_est"] = slot_plan["n_readable"].fillna(0).map(band_count)
slot_plan["has_16_in_band"] = slot_plan["central_band_count_est"].ge(N_SLICES)

slice_feasibility = (
    slot_plan.groupby("slot")
    .agg(
        n_slot_series=("SeriesInstanceUID", "size"),
        median_readable=("n_readable", "median"),
        median_band_count=("central_band_count_est", "median"),
        fraction_has_16_in_band=("has_16_in_band", "mean"),
    )
    .reset_index()
)

display(slice_feasibility)

## 5. Audit 130 mm physical-crop feasibility

In [ ]:
row_spacing_col = next((c for c in ["first_PixelSpacingRow", "PixelSpacingRow"] if c in slot_plan.columns), None)
col_spacing_col = next((c for c in ["first_PixelSpacingCol", "PixelSpacingCol"] if c in slot_plan.columns), None)
rows_col = next((c for c in ["first_Rows", "Rows"] if c in slot_plan.columns), None)
cols_col = next((c for c in ["first_Columns", "Columns"] if c in slot_plan.columns), None)

print("Geometry columns:")
print(" row spacing:", row_spacing_col)
print(" col spacing:", col_spacing_col)
print(" rows:", rows_col)
print(" cols:", cols_col)

crop_audit = slot_plan[["StudyInstanceUID", "SeriesInstanceUID", "slot"]].copy()

if all(x is not None for x in [row_spacing_col, col_spacing_col, rows_col, cols_col]):
    sr = pd.to_numeric(slot_plan[row_spacing_col], errors="coerce")
    sc = pd.to_numeric(slot_plan[col_spacing_col], errors="coerce")
    nr = pd.to_numeric(slot_plan[rows_col], errors="coerce")
    nc = pd.to_numeric(slot_plan[cols_col], errors="coerce")

    crop_audit["spacing_row_mm"] = sr
    crop_audit["spacing_col_mm"] = sc
    crop_audit["rows"] = nr
    crop_audit["cols"] = nc
    crop_audit["crop_rows_px"] = CROP_MM / sr
    crop_audit["crop_cols_px"] = CROP_MM / sc

    valid = (
        sr.gt(0) & sc.gt(0) & nr.gt(0) & nc.gt(0)
        & np.isfinite(crop_audit["crop_rows_px"])
        & np.isfinite(crop_audit["crop_cols_px"])
    )
    crop_audit["physical_crop_metadata_valid"] = valid
    crop_audit["crop_fits_image"] = (
        valid
        & crop_audit["crop_rows_px"].le(nr)
        & crop_audit["crop_cols_px"].le(nc)
    )

    crop_summary = (
        crop_audit.groupby("slot")
        .agg(
            n=("SeriesInstanceUID", "size"),
            fraction_metadata_valid=("physical_crop_metadata_valid", "mean"),
            fraction_crop_fits=("crop_fits_image", "mean"),
            median_crop_rows_px=("crop_rows_px", "median"),
            median_crop_cols_px=("crop_cols_px", "median"),
        )
        .reset_index()
    )
else:
    crop_audit["physical_crop_metadata_valid"] = False
    crop_audit["crop_fits_image"] = False
    crop_summary = pd.DataFrame({
        "slot": [s for _,_,s in SLOTS],
        "n": 0,
        "fraction_metadata_valid": 0.0,
        "fraction_crop_fits": 0.0,
        "median_crop_rows_px": np.nan,
        "median_crop_cols_px": np.nan,
    })

display(crop_summary)

## 6. Public DINOv3 checkpoint provenance audit

This checks what the checkpoint itself can prove. Absence of UID/fold metadata means provenance is **not independently verifiable from the checkpoint**.

In [ ]:
CHECKPOINT_AUDIT_COLUMNS = [
    "fold",
    "path",
    "size_mb",
    "sha256",
    "load_ok",
    "object_type",
    "top_level_keys",
    "n_state_keys",
    "has_uid_metadata",
    "has_fold_metadata",
    "has_optimizer_state",
    "has_vit_signature",
    "has_dino_signature",
    "backbone_hint",
    "pool_hint",
    "provenance_verdict",
    "error",
]


def summarize_checkpoint(
    path,
    fold=None,
):
    row = {
        "fold": fold,
        "path": str(path),
        "size_mb": (
            path.stat().st_size
            / (1024 ** 2)
        ),
        "sha256": file_sha256(path),
        "load_ok": False,
        "object_type": None,
        "top_level_keys": None,
        "n_state_keys": None,
        "has_uid_metadata": False,
        "has_fold_metadata": False,
        "has_optimizer_state": False,
        "has_vit_signature": False,
        "has_dino_signature": False,
        "backbone_hint": None,
        "pool_hint": None,
        "provenance_verdict": None,
        "error": None,
    }

    try:
        obj = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        row["load_ok"] = True
        row["object_type"] = (
            type(obj).__name__
        )

        state = None

        if isinstance(obj, dict):

            keys = list(obj.keys())

            row["top_level_keys"] = (
                " | ".join(
                    map(str, keys[:60])
                )
            )

            low_keys = [
                str(k).lower()
                for k in keys
            ]

            uid_terms = (
                "uid",
                "study",
                "train_ids",
                "train_uid",
                "val_ids",
                "valid_ids",
                "validation_ids",
                "holdout",
            )

            row["has_uid_metadata"] = any(
                any(
                    term in key
                    for term in uid_terms
                )
                for key in low_keys
            )

            row["has_fold_metadata"] = any(
                "fold" in key
                for key in low_keys
            )

            row["has_optimizer_state"] = any(
                "optim" in key
                for key in low_keys
            )

            for container_key in [
                "state_dict",
                "model_state_dict",
                "model",
                "net",
                "weights",
            ]:
                if (
                    container_key in obj
                    and isinstance(
                        obj[container_key],
                        dict,
                    )
                ):
                    state = obj[
                        container_key
                    ]
                    break

            if state is None:
                tensor_like = {
                    k: v
                    for k, v in obj.items()
                    if torch.is_tensor(v)
                }

                if tensor_like:
                    state = tensor_like

        elif hasattr(
            obj,
            "state_dict",
        ):
            state = obj.state_dict()

        if state is not None:

            skeys = list(
                state.keys()
            )

            row["n_state_keys"] = len(
                skeys
            )

            joined = "\n".join(
                map(
                    str,
                    skeys[:10000],
                )
            ).lower()

            vit_terms = [
                "patch_embed",
                "pos_embed",
                "cls_token",
                "blocks.",
                "attn.qkv",
            ]

            row["has_vit_signature"] = (
                sum(
                    term in joined
                    for term in vit_terms
                )
                >= 2
            )

            row["has_dino_signature"] = any(
                term in joined
                for term in [
                    "dinov3",
                    "register_tokens",
                    "mask_token",
                    "backbone.blocks",
                ]
            )

            if row[
                "has_vit_signature"
            ]:
                row[
                    "backbone_hint"
                ] = "ViT-like"

            if row[
                "has_dino_signature"
            ]:
                row[
                    "backbone_hint"
                ] = "DINO/ViT-like"

            if "slot" in joined:
                row[
                    "pool_hint"
                ] = "slot_embedding"

            if any(
                term in joined
                for term in [
                    "label_att",
                    "labelattention",
                    "query",
                    "cross_attn",
                ]
            ):
                if row[
                    "pool_hint"
                ]:
                    row[
                        "pool_hint"
                    ] += "+target_attention"
                else:
                    row[
                        "pool_hint"
                    ] = "target_attention"

        # Critical rule:
        # architecture signatures are NOT provenance.
        if row["has_uid_metadata"]:
            row[
                "provenance_verdict"
            ] = (
                "POSSIBLE_UID_PROVENANCE_PRESENT_REQUIRES_MANUAL_REVIEW"
            )
        else:
            row[
                "provenance_verdict"
            ] = (
                "TRAIN_UID_PROVENANCE_NOT_VERIFIABLE_FROM_CHECKPOINT"
            )

    except Exception as e:

        row["error"] = (
            f"{type(e).__name__}: {e}"
        )

        row[
            "provenance_verdict"
        ] = "CHECKPOINT_LOAD_FAILED"

    return row


checkpoint_records = [
    summarize_checkpoint(
        path,
        fold,
    )
    for fold, path
    in PUBLIC_CHECKPOINT_PATHS
]


checkpoint_audit = pd.DataFrame(
    checkpoint_records,
    columns=CHECKPOINT_AUDIT_COLUMNS,
)


print(
    "Checkpoint candidates audited:",
    len(checkpoint_audit),
)


if len(checkpoint_audit):
    display(
        checkpoint_audit
    )
else:
    print(
        "⚠️ Public checkpoint provenance unavailable; "
        "clean Phase 4B-2B will not depend on these weights."
    )

## 7. Freeze the Phase 4B-2B representation specification

In [ ]:
all_public_provenance_verifiable = bool(
    len(checkpoint_audit) > 0
    and checkpoint_audit[
        "has_uid_metadata"
    ].fillna(False).all()
)

# Even if UID-like metadata exists, this notebook does not approve
# competition-fine-tuned public weights automatically.
public_weights_clean_gold_approved = False


representation_spec = {
    "experiment_version":
        EXPERIMENT_VERSION,

    "phase4b2a_complete": True,

    "next_experiment":
        "PHASE4B2B_CLEAN_SERIES_AWARE_TARGET_ATTENTION",

    "explicit_kaggle_inputs": {
        "phase4b1a_dir":
            str(PHASE4B1A_DIR),

        "phase4b1b_v1_dir":
            str(PHASE4B1B_V1_DIR),

        "pipeline_json":
            str(PIPELINE_JSON_PATH),

        "series_index":
            str(SERIES_INDEX_PATH),

        "slice_selection":
            str(SLICE_SELECTION_PATH),

        "study_index":
            str(STUDY_INDEX_PATH),

        "baseline_v1_json":
            str(BASELINE_V1_JSON_PATH),
    },

    "baseline_v1_reference": {
        "experiment_version":
            baseline_v1_meta.get(
                "experiment_version"
            ),

        "macro_roc_auc":
            baseline_v1_meta.get(
                "gold_results",
                {},
            ).get(
                "macro_roc_auc"
            ),

        "macro_average_precision":
            baseline_v1_meta.get(
                "gold_results",
                {},
            ).get(
                "macro_average_precision"
            ),
    },

    "public_notebook_principles_adopted": {
        "six_slots": True,

        "slots": [
            {
                "plane": plane,
                "fat_suppression": int(fs),
                "slot": slot,
            }
            for plane, fs, slot
            in SLOTS
        ],

        "physical_crop_mm":
            CROP_MM,

        "image_size":
            MODEL_SIZE,

        "slice_band":
            list(SLICE_BAND),

        "n_slices_per_slot":
            N_SLICES,

        "series_type_embedding":
            True,

        "target_specific_attention":
            True,

        "rank_ensemble_later":
            True,
    },

    "public_notebook_components_not_adopted_now": [
        "public-leaderboard tuned target blend constants",
        "adaptive second-pass leaderboard calibration",
        "large multi-family ensemble",
        "competition-finetuned public DINO weights as a clean benchmark without verified train UID provenance",
    ],

    "checkpoint_inventory_count":
        int(
            len(
                checkpoint_inventory
            )
        ),

    "public_checkpoint_candidates_audited":
        int(
            len(
                checkpoint_audit
            )
        ),

    "public_fold_train_uid_provenance_verifiable_from_checkpoint":
        all_public_provenance_verifiable,

    "public_competition_weights_clean_gold_approved":
        public_weights_clean_gold_approved,

    "clean_validation_rule":
        (
            "58 gold studies remain excluded from "
            "architecture/loss/epoch selection."
        ),

    "recommended_phase4b2b_backbone_policy":
        (
            "Use a generic self-supervised pretrained encoder "
            "with no competition-label exposure. "
            "Public competition-finetuned fold weights may be "
            "studied for architecture/provenance but are not "
            "approved for the clean 58-gold benchmark."
        ),

    "slot_coverage":
        slot_coverage.to_dict(
            "records"
        ),

    "slice_feasibility":
        slice_feasibility.to_dict(
            "records"
        ),

    "physical_crop_summary":
        crop_summary.to_dict(
            "records"
        ),
}


SPEC_JSON = (
    OUT_DIR
    / "PHASE4B2A_DINOV3_REPRESENTATION_AUDIT_v1.json"
)

SLOT_CSV = (
    OUT_DIR
    / "rsna_phase4b2a_six_slot_plan_v1.csv"
)

SLOT_COVERAGE_CSV = (
    OUT_DIR
    / "rsna_phase4b2a_slot_coverage_v1.csv"
)

SLICE_FEAS_CSV = (
    OUT_DIR
    / "rsna_phase4b2a_slice_feasibility_v1.csv"
)

CROP_CSV = (
    OUT_DIR
    / "rsna_phase4b2a_physical_crop_audit_v1.csv"
)

CKPT_INVENTORY_CSV = (
    OUT_DIR
    / "rsna_phase4b2a_checkpoint_inventory_v1.csv"
)

CKPT_CSV = (
    OUT_DIR
    / "rsna_phase4b2a_public_dino_checkpoint_audit_v1.csv"
)


slot_plan.to_csv(
    SLOT_CSV,
    index=False,
)

slot_coverage.to_csv(
    SLOT_COVERAGE_CSV,
    index=False,
)

slice_feasibility.to_csv(
    SLICE_FEAS_CSV,
    index=False,
)

crop_audit.to_csv(
    CROP_CSV,
    index=False,
)

checkpoint_inventory.to_csv(
    CKPT_INVENTORY_CSV,
    index=False,
)

checkpoint_audit.to_csv(
    CKPT_CSV,
    index=False,
)


with open(
    SPEC_JSON,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        representation_spec,
        f,
        indent=2,
        ensure_ascii=False,
    )


print()
print(
    "✅ PHASE 4B-2A COMPLETE"
)

print()
print(
    "Saved:"
)

for p in [
    SPEC_JSON,
    SLOT_CSV,
    SLOT_COVERAGE_CSV,
    SLICE_FEAS_CSV,
    CROP_CSV,
    CKPT_INVENTORY_CSV,
    CKPT_CSV,
]:
    print(
        " -",
        p.name,
    )

print()
print(
    "Public competition weights approved "
    "for clean gold evaluation:",
    public_weights_clean_gold_approved,
)

# STOP GATE — Phase 4B-2A

This notebook is considered **finished** if it reaches:

```text
✅ PHASE 4B-2A COMPLETE
```

Public DINO checkpoints are optional for completing the representation audit.

If their train/validation UID provenance cannot be proven, they are simply marked:

```text
NOT APPROVED FOR CLEAN GOLD EVALUATION
```

That does **not** block our next model.

## Send back

```text
PHASE4B2A_DINOV3_REPRESENTATION_AUDIT_v1.json

rsna_phase4b2a_slot_coverage_v1.csv
rsna_phase4b2a_slice_feasibility_v1.csv
rsna_phase4b2a_physical_crop_audit_v1.csv

rsna_phase4b2a_checkpoint_inventory_v1.csv
rsna_phase4b2a_public_dino_checkpoint_audit_v1.csv

rsna_phase4b2a_six_slot_plan_v1.csv

executed notebook
```

## Then we start Phase 4B-2B

The first clean stronger model will use:

```text
6 plane × contrast slots
        +
ordered multi-slice context
        +
physical-FOV normalization where valid
        +
series/slot embeddings
        +
12 target-specific attention queries
        +
PSEUDO_LABELS_v1_1
        +
cv_group_id leakage protection
        +
macro-target soft-label loss
```

We will use a **clean pretrained encoder** with no competition-label exposure for the 58-gold benchmark.